# Graph-Based Fraud Detection with PMP

I examined how well Partitioning Message Passing (PMP) identifies illicit Bitcoin transactions in periods later than those used for training. I compared it with GCN, GAT, and BWGNN on Elliptic, a dataset not evaluated in the PMP paper. I report the paper's published benchmarks separately from my Elliptic results and from a check of one published Amazon checkpoint.

[Original PMP paper](https://arxiv.org/html/2412.00020v1) · [Elliptic paper](https://arxiv.org/abs/1908.02591) · [Elliptic dataset](https://www.kaggle.com/datasets/ellipticco/elliptic-data-set) · [BWGNN reference implementation](https://github.com/squareRoot3/Rethinking-Anomaly-Detection)

## Implementation note

This notebook uses the authors’ original implementation of PMP, with changes to support a CPU-only environment and to load and evaluate the Elliptic Bitcoin dataset. These changes do not alter PMP’s model architecture or its partitioning message-passing mechanism.

[Official PMP repository](https://github.com/Xtra-Computing/PMP) · [Source revision used in this notebook (`3f7629f`)](https://github.com/Xtra-Computing/PMP/tree/3f7629f6c180891a0bc1bba3c66d94d288a1ddae)


## 1. Published benchmark results

I took these values from Tables 1 and 2 of the PMP paper. They are the authors' means over ten runs with a 40% training, 20% validation, and 40% test split, converted from a 0–100 scale to a 0–1 scale. I include them as a reference. They are not outputs from retraining on the original datasets in this notebook.

| Model | T-Finance ROC-AUC | T-Finance Macro-F1 | YelpChi ROC-AUC | Amazon ROC-AUC |
|---|---:|---:|---:|---:|
| GCN | 0.9331 | 0.8686 | 0.5983 | 0.8369 |
| GAT | 0.9277 | 0.6219 | 0.5715 | 0.8102 |
| BWGNN | 0.9584 | 0.8866 | 0.9054 | 0.9742 |
| PMP | 0.9710 | 0.9190 | 0.9397 | 0.9757 |

[Source: PMP paper, Tables 1 and 2](https://proceedings.iclr.cc/paper_files/paper/2024/file/9251fa7fcc356bbfc16c040f4c030d83-Paper-Conference.pdf#page=7)


## 2. Runtime and comparison models

I ran the experiments on CPU in an isolated Python 3.10 environment. I used DGL layers for the GCN and GAT baselines and the authors' reference implementation for BWGNN. The Elliptic experiment uses its own split and training settings, so it is a separate evaluation rather than a reproduction of the paper's benchmark protocol.


In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

def run(*args, **kwargs):
    return subprocess.run([str(a) for a in args], check=True, **kwargs)

if shutil.which("uv"):
    UV = [shutil.which("uv")]
else:
    run(sys.executable, "-m", "pip", "install", "--quiet", "uv")
    UV = [sys.executable, "-m", "uv"]

work = Path(tempfile.mkdtemp(prefix="pmp-cpu-"))
venv = work / "venv"
run(*UV, "python", "install", "3.10.22")
run(*UV, "venv", "--python", "3.10.22", venv)
py = venv / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
install = [*UV, "pip", "install", "--python", py]
run(*install, "--index", "https://download.pytorch.org/whl/cpu", "torch==2.0.1+cpu")
run(*install, "--find-links", "https://data.pyg.org/whl/torch-2.0.0+cpu.html",
    "torch_scatter==2.1.1+pt20cpu", "torch_sparse==0.6.17+pt20cpu",
    "dgl==1.1.1", "torch_geometric==2.3.1", "numpy==1.23.5",
    "scipy==1.10.0", "scikit-learn==1.2.1", "matplotlib==3.7.0",
    "PyYAML==6.0.1", "networkx==2.8.4", "ogb==1.3.6",
    "torchmetrics==0.11.4", "tensorboard==2.21.0", "setuptools==80.9.0",
    "tqdm==4.70.1", "pandas==2.3.3")
env = dict(os.environ, DGLBACKEND="pytorch", OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2")
run(py, "-c", "import torch, dgl; print('PyTorch', torch.__version__, '| DGL', dgl.__version__)", env=env)

In [ ]:
from urllib.request import urlopen, Request
from zipfile import ZipFile, is_zipfile
from time import sleep

LOCAL_ELLIPTIC_ZIP = ""

def fetch(url, dest):
    for attempt in range(2):
        try:
            with urlopen(Request(url, headers={"User-Agent": "Mozilla/5.0"}), timeout=120) as response:
                with dest.open("wb") as stream:
                    shutil.copyfileobj(response, stream)
            if not is_zipfile(dest):
                raise ValueError(f"The download for {dest.name} was not a ZIP file")
            return
        except (OSError, ValueError):
            if attempt:
                raise
            sleep(2)

commit = "3f7629f6c180891a0bc1bba3c66d94d288a1ddae"
pmp_zip = work / "pmp.zip"
fetch(f"https://codeload.github.com/Xtra-Computing/PMP/zip/{commit}", pmp_zip)
with ZipFile(pmp_zip) as z:
    z.extractall(work)
repo = work / f"PMP-{commit}"
dataset_dir = repo / "datasets"

if LOCAL_ELLIPTIC_ZIP:
    data_zip = Path(LOCAL_ELLIPTIC_ZIP).expanduser().resolve()
    if not is_zipfile(data_zip):
        raise ValueError(f"Not a ZIP file: {data_zip}")
else:
    data_zip = work / "elliptic.zip"
    print("Downloading the Elliptic data (about 147 MiB)...", flush=True)
    try:
        fetch("https://www.kaggle.com/api/v1/datasets/download/ellipticco/elliptic-data-set", data_zip)
    except (OSError, ValueError) as exc:
        raise RuntimeError("The Elliptic dataset ZIP could not be downloaded.") from exc
print("Source files ready.")


## 3. Elliptic experiment design

- Each node represents a Bitcoin transaction. A directed edge indicates that an output from one transaction was used as an input to another. The positive class is *illicit*, which does not necessarily mean confirmed fraud.
- I used the same 93 local transaction features for all models. I fitted feature standardization using only the training data. Transactions without labels remained in the graph but did not contribute to the loss or evaluation metrics.
- I split the data chronologically. Time steps 1–30 were used for training, 31–34 for validation, and 35–49 for testing. I selected the checkpoint by validation average precision (AP). When a decision threshold was needed, I selected it by positive-class F1 on the validation set. Test ROC-AUC does not depend on that threshold.
- GCN and GAT used a directed graph with self-loops. BWGNN used an undirected projection for its spectral filters, while PMP used the directed graph. This design compares complete model implementations, not the effect of changing one architectural component.
- I trained each model from scratch using the same ten seeds (1234–1243) and the same time split. I report the mean and sample standard deviation of test ROC-AUC across these runs.


In [ ]:
patch_source = (
    'from pathlib import Path\n'
    'import yaml\n'
    '\n'
    'def replace(file, old, new, count=1):\n'
    '    p = Path(file)\n'
    "    s = p.read_text(encoding='utf-8')\n"
    '    assert s.count(old) == count, (file, old, s.count(old))\n'
    "    p.write_text(s.replace(old, new), encoding='utf-8')\n"
    '\n'
    "replace('main.py', '    torch.cuda.set_device(args.gpu_id)', '    torch.set_num_threads(2)')\n"
    "replace('main.py', 'import torch\\n', 'import torch\\nimport dgl\\n')\n"
    'replace(\'main.py\', "        set_random_seed(config[\'seed\'])", "        set_random_seed(config[\'seed\'])\\n        dgl.seed(config[\'seed\'])")\n'
    "replace('main.py', 'np.std(metric_list, axis=0, ddof=1)', 'np.std(metric_list, axis=0, ddof=1) if len(metric_list) > 1 else np.nan')\n"
    "replace('training_procedure/prepare.py', '.cuda()', '', count=2)\n"
    "replace('training_procedure/train.py', 'torch.cuda.current_device()', 'next(model.parameters()).device')\n"
    "replace('training_procedure/evaluate.py', 'torch.cuda.current_device()', 'next(model.parameters()).device')\n"
    "replace('training_procedure/evaluate.py', 'batch_seq.cuda()', 'batch_seq.to(next(model.parameters()).device)')\n"
    "replace('training_procedure/evaluate.py', '''    F1 = 2 * precision * recall / (precision + recall)\n"
    "    idx = F1.argmax(axis=0)''', '''    # The last PR point does not have a corresponding threshold.\n"
    '    p, r = precision[:-1], recall[:-1]\n'
    '    F1 = np.divide(2 * p * r, p + r, out=np.zeros_like(p), where=(p + r) > 0)\n'
    "    idx = F1.argmax(axis=0)''')\n"
    'replace(\'DataHelper/datasetHelper.py\', \'num_workers=8)\', "num_workers=self.config[\'num_workers\'])", count=2)\n'
    'replace(\'DataHelper/datasetHelper.py\', "        elif self.dataset_name in [\'tfinance\', \'tsocial\']:", \'\'\'        elif self.dataset_name == \'elliptic\':\n'
    "            dataset, _ = load_graphs(osp.join(self.dataset_source_folder_path, 'elliptic.bin'))\n"
    '\n'
    "        elif self.dataset_name in ['tfinance', 'tsocial']:''')\n"
    'replace(\'DataHelper/datasetHelper.py\', "self.num_classes     = dataset.num_classes if not self.dataset_name in [\'tfinance\', \'tsocial\', \'grab\'] else len(torch.unique(data.ndata[\'label\']))",\n'
    '        "self.num_classes     = 2 if self.dataset_name == \'elliptic\' else (dataset.num_classes if not self.dataset_name in [\'tfinance\', \'tsocial\', \'grab\'] else len(torch.unique(data.ndata[\'label\'])))")\n'
    "replace('DataHelper/datasetHelper.py', 'torch.sum(self.labels):>6', '(self.labels == 1).sum():>6')\n"
    "config = yaml.safe_load(Path('config/amazon.yml').read_text(encoding='utf-8'))\n"
    "config['dataset'] = 'elliptic'\n"
    "config['LA-SAGE-S'].update(epochs=20, patience=6, monitor='ap_gnn', norm_feat=False, hid_dim=128,\n"
    '    batch_size=512, test_batch_size=2048, dropout=0.2, lr=0.003, full_neighbors=True,\n'
    '    weighted_loss=True, add_self_loop=False)\n'
    "Path('config/elliptic.yml').write_text(yaml.safe_dump(config), encoding='utf-8')\n"
)
patch_file = work / "adapt.py"
patch_file.write_text(patch_source, encoding="utf-8")
run(py, patch_file, cwd=repo, env=env)
# Allow the same explicit list of experiment seeds in all four models.
main_path = repo / "main.py"
main_code = main_path.read_text(encoding="utf-8")
old_seed_flag = "choices = [0, 1, 1234]"
assert main_code.count(old_seed_flag) == 1
main_code = main_code.replace(old_seed_flag, "")
old_call = "        model, best_dev_results, final_test_results, loss = main(args"
assert main_code.count(old_call) == 1
main_code = main_code.replace(
    old_call, "        print('RESULT_SEED', config['seed'], flush=True)\n" + old_call
)
main_path.write_text(main_code, encoding="utf-8")


In [ ]:
prep_source = (
    'from pathlib import Path\n'
    'from zipfile import ZipFile\n'
    'import json\n'
    'import sys\n'
    'import dgl\n'
    'import numpy as np\n'
    'import pandas as pd\n'
    'import torch\n'
    'from sklearn.preprocessing import StandardScaler\n'
    'from sklearn.linear_model import LogisticRegression\n'
    'from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve, f1_score, precision_score, recall_score\n'
    '\n'
    'source, output = map(Path, sys.argv[1:])\n'
    'output.mkdir(exist_ok=True, parents=True)\n'
    'with ZipFile(source) as z:\n'
    '    def csv(name, **kwargs):\n'
    "        member, = [p for p in z.namelist() if p.rsplit('/', 1)[-1] == name]\n"
    '        with z.open(member) as f:\n'
    '            return pd.read_csv(f, **kwargs)\n'
    '\n'
    "    x = csv('elliptic_txs_features.csv', header=None, usecols=range(95),\n"
    '            dtype={i: np.float32 for i in range(2, 95)})\n'
    "    classes = csv('elliptic_txs_classes.csv', dtype={'class': str}).set_index('txId')\n"
    "    edges = csv('elliptic_txs_edgelist.csv')\n"
    '\n'
    "ids = pd.Index(x.pop(0), name='txId')\n"
    'time = x.pop(1).to_numpy()\n'
    'assert ids.is_unique\n'
    "raw_y = classes.reindex(ids)['class']\n"
    "assert set(raw_y.unique()) == {'1', '2', 'unknown'}\n"
    "y = raw_y.map({'1': 1, '2': 0, 'unknown': 2}).to_numpy(dtype=np.int64)\n"
    "src = ids.get_indexer(edges['txId1'])\n"
    "dst = ids.get_indexer(edges['txId2'])\n"
    'assert (src >= 0).all() and (dst >= 0).all()\n'
    "assert (time[src] == time[dst]).all(), 'Unexpected connections across time windows'\n"
    '\n'
    'train = (time <= 30) & (y != 2)\n'
    'val = (time >= 31) & (time <= 34) & (y != 2)\n'
    'test = (time >= 35) & (y != 2)\n'
    'assert not ((train & val) | (val & test) | (train & test)).any()\n'
    'for mask in [train, val, test]:\n'
    '    assert set(y[mask]) == {0, 1}\n'
    '\n'
    '# Use only the 93 local attributes other than the time index. Standardize\n'
    '# using training rows only. Both models receive the same attributes.\n'
    'xx = x.to_numpy(dtype=np.float32)\n'
    'xx = StandardScaler().fit(xx[train]).transform(xx)\n'
    'assert np.isfinite(xx).all()\n'
    '\n'
    'g = dgl.graph((src, dst), num_nodes=len(y))\n'
    "g.ndata['feature'] = torch.from_numpy(xx)\n"
    "g.ndata['label'] = torch.from_numpy(y)\n"
    "for key, mask in zip(('train_mask', 'val_mask', 'test_mask'), (train, val, test)):\n"
    '    g.ndata[key] = torch.from_numpy(mask)\n'
    "dgl.save_graphs(str(output / 'elliptic.bin'), [g])\n"
    'np.save(output / "elliptic_times.npy", time)\n'
    '\n'
    'proxy_labels = np.where(train, y, 2)\n'
    'test_neighbor_train_labels = int(((proxy_labels[src] != 2) & test[dst]).sum())\n'
    'assert test_neighbor_train_labels == 0\n'
    'stats = dict(nodes=len(y), edges=len(edges), local_features=xx.shape[1], unlabeled=int((y==2).sum()),\n'
    '             test_neighbor_train_labels=test_neighbor_train_labels)\n'
    "for key, mask in [('train', train), ('validation', val), ('test', test)]:\n"
    "    stats[key] = {'labeled': int(mask.sum()), 'illicit': int((y[mask] == 1).sum())}\n"
    "print('DATASET:', json.dumps(stats), flush=True)\n"
    '\n'
    '# Non-graph baseline. Threshold chosen on validation only, maximizing F1.\n'
    "lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=1234)\n"
    'lr.fit(xx[train], y[train])\n'
    'p, r, th = precision_recall_curve(y[val], lr.predict_proba(xx[val])[:, 1])\n'
    'p, r = p[:-1], r[:-1]\n'
    'f = np.divide(2*p*r, p+r, out=np.zeros_like(p), where=(p+r)>0)\n'
    'scores = lr.predict_proba(xx[test])[:, 1]\n'
    'pred = scores >= th[f.argmax()]\n'
    "baseline = {'ROC-AUC': roc_auc_score(y[test], scores), 'AP': average_precision_score(y[test], scores),\n"
    "            'Illicit F1': f1_score(y[test], pred), 'Precision': precision_score(y[test], pred),\n"
    "            'Recall': recall_score(y[test], pred)}\n"
    "print('BASELINE:', json.dumps(baseline), flush=True)\n"
    "(output / 'comparison_inputs.json').write_text(json.dumps({'dataset': stats, 'baseline': baseline}, indent=2), encoding='utf-8')\n"
)
prep_file = work / "prepare_elliptic.py"
prep_file.write_text(prep_source, encoding="utf-8")
run(py, "-u", prep_file, data_zip, dataset_dir, cwd=repo, env=env)

## 4. Baseline implementations


In [ ]:
from hashlib import sha256

bw_ref = work / "BWGNN_reference.py"
bw_version = "9b0bf96c417603290b2c5b0e6e567b52fd7eadea"
with urlopen(f"https://raw.githubusercontent.com/squareRoot3/Rethinking-Anomaly-Detection/{bw_version}/BWGNN.py",
             timeout=60) as f:
    bw_ref.write_bytes(f.read())
assert sha256(bw_ref.read_bytes()).hexdigest() == "0ac45b1104f80fa17c50cd5e3bf9aa6241e008e6b57fdab9366ec69f80cb2f84"

In [ ]:
baseline_source = (
    '"""Train graph baselines with the same Elliptic split and features as PMP."""\n'
    'import copy\n'
    'import json\n'
    'from pathlib import Path\n'
    'import sys\n'
    '\n'
    'import dgl\n'
    'import numpy as np\n'
    'import torch\n'
    'import torch.nn as nn\n'
    'import torch.nn.functional as F\n'
    'from dgl.nn import GraphConv, GATConv\n'
    'from sklearn.metrics import (average_precision_score, f1_score, precision_recall_curve,\n'
    '                             precision_score, recall_score, roc_auc_score)\n'
    'from BWGNN_reference import BWGNN\n'
    '\n'
    'data_dir = Path(sys.argv[1])\n'
    'g = dgl.load_graphs(str(data_dir / "elliptic.bin"))[0][0]\n'
    'torch.set_num_threads(2)\n'
    'SEED = int(sys.argv[2])\n'
    'HIDDEN = 32\n'
    'MAX_EPOCHS = 40\n'
    'PATIENCE = 8\n'
    'LR = 0.003\n'
    '\n'
    '# There are no edges between time steps. Induce phase graphs on the\n'
    '# whole time window, including transactions whose labels are unknown.\n'
    'times = np.load(data_dir / "elliptic_times.npy")\n'
    'phase_nodes = [times <= 30, (times >= 31) & (times <= 34), times >= 35]\n'
    "mask_names = ['train_mask', 'val_mask', 'test_mask']\n"
    'split = [dgl.node_subgraph(g, torch.from_numpy(m)) for m in phase_nodes]\n'
    'for sg, m in zip(split, mask_names):\n'
    "    assert set(sg.ndata['label'][sg.ndata[m]].tolist()) == {0, 1}\n"
    'assert sum(sg.num_edges() for sg in split) == g.num_edges()\n'
    "# GraphConv and GATConv use self-loops. BWGNN's spectral Laplacian is\n"
    '# evaluated on the undirected projection, with self-information in its filters.\n'
    'directed = [dgl.add_self_loop(dgl.remove_self_loop(sg)) for sg in split]\n'
    'spectral = []\n'
    'for sg in split:\n'
    '    h = dgl.to_bidirected(dgl.remove_self_loop(sg), copy_ndata=True)\n'
    '    spectral.append(h)\n'
    'del g, split\n'
    "n_features = directed[0].ndata['feature'].shape[1]\n"
    "ytrain = directed[0].ndata['label'][directed[0].ndata['train_mask'].bool()]\n"
    'weight = torch.tensor([1.0, (ytrain == 0).sum().item() / (ytrain == 1).sum().item()])\n'
    '\n'
    'class GCN(nn.Module):\n'
    '    def __init__(self):\n'
    '        super().__init__()\n'
    '        self.first = GraphConv(n_features, HIDDEN)\n'
    '        self.second = GraphConv(HIDDEN, 2)\n'
    '\n'
    '    def forward(self, g, x):\n'
    '        return self.second(g, F.dropout(F.relu(self.first(g, x)), p=0.2, training=self.training))\n'
    '\n'
    'class GAT(nn.Module):\n'
    '    def __init__(self):\n'
    '        super().__init__()\n'
    '        self.first = GATConv(n_features, 8, 4, feat_drop=0.2, attn_drop=0.2)\n'
    '        self.second = GATConv(32, 2, 1, feat_drop=0.2)\n'
    '\n'
    '    def forward(self, g, x):\n'
    '        return self.second(g, F.elu(self.first(g, x)).flatten(1)).squeeze(1)\n'
    '\n'
    'def fit(name, build, graphs):\n'
    '    np.random.seed(SEED)\n'
    '    torch.manual_seed(SEED)\n'
    '    dgl.seed(SEED)\n'
    '    tr, va, te = graphs\n'
    '    model = build()\n'
    '    opt = torch.optim.Adam(model.parameters(), lr=LR)\n'
    '    def forward(h):\n'
    "        return (model.testlarge(h, h.ndata['feature']) if name == 'BWGNN'\n"
    "                else model(h, h.ndata['feature']))\n"
    '\n'
    '    @torch.no_grad()\n'
    '    def evaluate(h, mask):\n'
    '        model.eval()\n'
    '        picked = h.ndata[mask].bool()\n'
    '        logits = forward(h)[picked]\n'
    "        return h.ndata['label'][picked].numpy(), logits.softmax(dim=-1)[:, 1].numpy()\n"
    '\n'
    '    best_ap, best_epoch, state = -1.0, 0, None\n'
    '    for epoch in range(1, MAX_EPOCHS + 1):\n'
    '        model.train()\n'
    '        opt.zero_grad()\n'
    "        loss = F.cross_entropy(forward(tr)[tr.ndata['train_mask'].bool()], ytrain, weight=weight)\n"
    '        loss.backward()\n'
    '        opt.step()\n'
    "        yt, score = evaluate(va, 'val_mask')\n"
    '        ap = average_precision_score(yt, score)\n'
    '        if ap > best_ap:\n'
    '            best_ap, best_epoch, state = ap, epoch, copy.deepcopy(model.state_dict())\n'
    '        if epoch - best_epoch >= PATIENCE:\n'
    '            break\n'
    '    model.load_state_dict(state)\n'
    "    yt, score = evaluate(va, 'val_mask')\n"
    '    p, r, th = precision_recall_curve(yt, score)\n'
    '    p, r = p[:-1], r[:-1]\n'
    '    f = np.divide(2*p*r, p+r, out=np.zeros_like(p), where=(p+r)>0)\n'
    "    yt, score = evaluate(te, 'test_mask')\n"
    '    pred = score >= th[f.argmax()]\n'
    "    out = {'ROC-AUC': roc_auc_score(yt, score), 'AP': average_precision_score(yt, score),\n"
    "           'Illicit F1': f1_score(yt, pred), 'Precision': precision_score(yt, pred, zero_division=0),\n"
    "           'Recall': recall_score(yt, pred), 'best_epoch': best_epoch}\n"
    '    print(name, json.dumps(out), flush=True)\n'
    '    return out\n'
    '\n'
    '\n'
    'name = sys.argv[3]\n'
    "models = {'GCN': (GCN, directed), 'GAT': (GAT, directed),\n"
    "          'BWGNN': (lambda: BWGNN(n_features, HIDDEN, 2, spectral[0], d=2), spectral)}\n"
    'fit(name, *models[name])\n'
)
train_baselines = work / "elliptic_baselines.py"
train_baselines.write_text(baseline_source, encoding="utf-8")


## 5. Training and evaluation on Elliptic


In [ ]:
import concurrent.futures
import json
import re
import statistics
import time

SEEDS = list(range(1234, 1244))
MODELS = ("GCN", "GAT", "BWGNN", "PMP")
MAX_PARALLEL = 2  # Use 1 on a low-memory CPU runtime.

results = work / "elliptic_10_runs"
results.mkdir(exist_ok=True)
experiment_env = dict(env, MKL_NUM_THREADS="2")

def run_elliptic_model(model, seed):
    path=results/f'{model}_{seed}.log'
    dat=path.with_suffix('.json')
    if model=='PMP':
        args=[py,'-u','main.py','--dataset','elliptic','--num_workers','0','--seed',str(seed),'--best_model_path',str(results/f'checkpoint-{seed}')]
    else:
        args=[py,'-u',str(train_baselines),str(dataset_dir),str(seed),model]
    start=time.monotonic()
    with path.open('w') as log: subprocess.run([str(x) for x in args],cwd=repo,env=experiment_env,stdout=log,stderr=subprocess.STDOUT,check=True)
    trace=path.read_text()
    if model=='PMP':
        assert f'RESULT_SEED {seed}' in trace
        score=float(re.search(r'Best Epoch Test is .*?\x27auc_gnn\x27: ([0-9.eE+-]+)',trace)[1])
    else:
        score=json.loads(re.search(r'^'+model+r' (\{.*\})$',trace,re.M)[1])['ROC-AUC']
    result=dict(model=model,seed=seed,auc=score,seconds=round(time.monotonic()-start,1))
    dat.write_text(json.dumps(result))
    return result


jobs = [(model, seed) for seed in SEEDS for model in MODELS]
scores = {model: {} for model in MODELS}
with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_PARALLEL) as pool:
    pending = {pool.submit(run_elliptic_model, model, seed): (model, seed)
               for model, seed in jobs}
    for i, future in enumerate(concurrent.futures.as_completed(pending), 1):
        model, seed = pending[future]
        try:
            row = future.result()
        except Exception as exc:
            log = results / f"{model}_{seed}.log"
            print(log.read_text(encoding="utf-8", errors="replace")[-5000:])
            raise RuntimeError(f"{model}, seed {seed} failed") from exc
        scores[model][seed] = row["auc"]
        print(f"{i:>2}/{len(jobs)}  {model:5}  seed={seed}  ROC-AUC={row['auc']:.4f}", flush=True)

assert all(set(vals) == set(SEEDS) for vals in scores.values())
print("\nElliptic Bitcoin — Test ROC-AUC over 10 runs per model")
print(f"{'Model':<7} {'Mean':>8} {'Sample SD':>10}")
for model, vals in scores.items():
    mean = statistics.mean(vals.values())
    sd = statistics.stdev(vals.values())
    print(f"{model:<7} {mean:8.4f} {sd:10.4f}")


 1/40  GCN    seed=1234  ROC-AUC=0.8422
 2/40  GAT    seed=1234  ROC-AUC=0.8233
 3/40  BWGNN  seed=1234  ROC-AUC=0.8927
 4/40  PMP    seed=1234  ROC-AUC=0.8511
 5/40  GCN    seed=1235  ROC-AUC=0.8374
 6/40  GCN    seed=1236  ROC-AUC=0.8168
 7/40  GAT    seed=1235  ROC-AUC=0.7800
 8/40  PMP    seed=1235  ROC-AUC=0.8552
 9/40  BWGNN  seed=1235  ROC-AUC=0.8891
10/40  GAT    seed=1236  ROC-AUC=0.7984
11/40  GCN    seed=1237  ROC-AUC=0.8319
12/40  BWGNN  seed=1236  ROC-AUC=0.8968
13/40  PMP    seed=1236  ROC-AUC=0.8601
14/40  GAT    seed=1237  ROC-AUC=0.8123
15/40  GCN    seed=1238  ROC-AUC=0.8236
16/40  PMP    seed=1237  ROC-AUC=0.8756
17/40  BWGNN  seed=1237  ROC-AUC=0.8891
18/40  GAT    seed=1238  ROC-AUC=0.8198
19/40  GCN    seed=1239  ROC-AUC=0.8491
20/40  BWGNN  seed=1238  ROC-AUC=0.8930
21/40  PMP    seed=1238  ROC-AUC=0.8730
22/40  BWGNN  seed=1239  ROC-AUC=0.8398
23/40  GAT    seed=1239  ROC-AUC=0.8241
24/40  GCN    seed=1240  ROC-AUC=0.8310
25/40  PMP    seed=1239  ROC-AUC=0.8777


### Results and interpretation

**Elliptic test ROC-AUC, mean ± sample standard deviation over ten runs per model**

| Model | ROC-AUC |
|---|---:|
| GCN | 0.8332 ± 0.0092 |
| GAT | 0.8086 ± 0.0174 |
| BWGNN | 0.8837 ± 0.0166 |
| PMP | 0.8648 ± 0.0145 |

BWGNN had the highest mean ROC-AUC on this split (0.8837), and PMP ranked second (0.8648). PMP was applicable to a dataset outside the original paper, but it did not retain its lead in this comparison. I did not test whether the difference is statistically significant or isolate the effect of PMP's neighbor partitioning. A single additional dataset cannot establish whether the benchmarks in the original paper were selectively chosen.

The test set contained 16,670 labeled transactions, of which 1,083 (6.50%) were labeled illicit. Elliptic has no edges between time steps. Under this chronological split, test transactions therefore had no labeled neighbors from the training period, and PMP treated their neighbors as unlabeled.


## Appendix. Amazon checkpoint evaluation

I loaded a PMP checkpoint published by the authors and evaluated it on the Amazon test set without retraining. The goal was to check that the published implementation could reproduce the documented score of that particular checkpoint.

| Result | ROC-AUC |
|---|---:|
| My evaluation of the saved checkpoint | 0.9779 |
| Score documented for the same checkpoint | 0.9779 |
| Mean across the ten runs reported in the paper | 0.9757 |

Agreement on one checkpoint is not a reproduction of the ten-run mean. The saved code output reports metrics on a 0–100 scale.

[Published log for the evaluated checkpoint](https://github.com/Xtra-Computing/PMP/blob/3f7629f6c180891a0bc1bba3c66d94d288a1ddae/checkpoints/val_0.2/LA-SAGE-S/amazon/2023-07-03_23-52-09/results.txt)


In [ ]:
RECHECK_PAPER = ["amazon"]


In [ ]:
repro_source = (
    'import json\n'
    'from pathlib import Path\n'
    'import sys\n'
    'from types import SimpleNamespace\n'
    '\n'
    'import dgl\n'
    'import numpy as np\n'
    'import torch\n'
    'from utils.utils import get_config, load_data\n'
    'from utils.random_seeder import set_random_seed\n'
    'from utils.logger import Logger\n'
    'from training_procedure import Trainer\n'
    '\n'
    'dataset, local_data = sys.argv[1:]\n'
    'options = []\n'
    "root = Path('checkpoints/val_0.2/LA-SAGE-S') / dataset\n"
    "preferred_run = dict(amazon='2023-07-03_23-52-09', yelp='2023-07-03_20-27-53', tfinance='2023-07-04_13-07-46')[dataset]\n"
    "paths = sorted(root.glob(f'*/{dataset}.yml')) if 'all' in options else [root / preferred_run / f'{dataset}.yml']\n"
    'torch.set_num_threads(2)\n'
    'cache = {}\n'
    'for config_path in paths:\n'
    '    config = get_config(config_path)\n'
    "    config.update(num_workers=0, model_name='LA-SAGE-S')\n"
    '    flags = SimpleNamespace(**config)\n'
    '    flags.data_dir = local_data\n'
    "    key = tuple(config.get(k) for k in ('dataset', 'dataset_seed', 'train_size', 'val_size', 'homo', 'add_self_loop', 'norm_feat', 'batch_size', 'test_batch_size', 'val_batch_size', 'full_neighbors'))\n"
    "    set_random_seed(config['seed'])\n"
    "    dgl.seed(config['seed'])\n"
    '    if key not in cache:\n'
    '        helper = load_data(flags, config)\n'
    '        helper.load()\n'
    '        cache[key] = helper\n'
    '    else:\n'
    '        helper = cache[key]\n'
    '    T = Trainer(config=config, args=flags, logger=Logger(mode=[print]))\n'
    '    model = T.prepare_model(helper)\n'
    "    preferred = config_path.parent / 'best_val_model_1234.pth'\n"
    "    model.load_state_dict(torch.load(preferred, map_location='cpu', weights_only=True))\n"
    "    vlabels, vprobs, vpred = T.evaluation(helper, helper.val_loader, model, threshold_moving=config['threshold_moving'], thres=config['thres'])\n"
    '    val = T.eval_model(vlabels, vprobs, vpred)\n'
    "    tlabels, tprobs, tpred = T.evaluation(helper, helper.test_loader, model, threshold_moving=config['threshold_moving'], thres=val.best_pr_thres)\n"
    '    test = T.eval_model(tlabels, tprobs, tpred)\n'
    "    print('RESULT', json.dumps(dict(dataset=dataset, run=config_path.parent.name,\n"
    '        recommended=config_path.parent.name==preferred_run, n_test=int(tlabels.size), auc=test.auc_gnn*100,\n'
    '        f1_macro=test.f1_macro*100, gmean=test.gmean_gnn*100, threshold=float(val.best_pr_thres))), flush=True)\n'
)
if RECHECK_PAPER:
    cache = repo / "published_data"
    cache.mkdir(exist_ok=True)
    repro_file = repo / "_evaluate_published_pmp.py"
    repro_file.write_text(repro_source, encoding="utf-8")
    for name in RECHECK_PAPER:
        assert name in {"amazon", "yelp", "tfinance"}, name
        if name in {"amazon", "yelp"}:
            repo_name, archive_name = {"amazon": ("FraudAmazon", "Amazon"), "yelp": ("FraudYelp", "YelpChi")}[name]
            from hashlib import sha1
            suffix = sha1(f"https://data.dgl.ai/dataset/{repo_name}.zip".encode()).hexdigest()[:8]
            raw_dir = cache / f"{name}_{suffix}"
            filename = archive_name + ".mat"
            if not (raw_dir / filename).is_file():
                raw_dir.mkdir(exist_ok=True)
                archive = work / (archive_name + ".zip")
                fetch(f"https://raw.githubusercontent.com/YingtongDou/CARE-GNN/master/data/{archive_name}.zip", archive)
                with ZipFile(archive) as z:
                    z.extract(filename, raw_dir)
        elif not (dataset_dir / "tfinance").is_file():
            archive = work / "tfinance.zip"
            fetch("https://drive.usercontent.google.com/download?id=1gAMek_5Dsd4xPD0o7z-FFD8Jwxr0gJR5&export=download&confirm=t", archive)
            with ZipFile(archive) as z:
                z.extract("tfinance", dataset_dir)
        run(py, "-u", repro_file, name, cache, cwd=repo, env=env)
else:
    print("Published checkpoint evaluation was not run.")


RESULT {"dataset": "amazon", "run": "2023-07-03_23-52-09", "recommended": true, "n_test": 3457, "auc": 97.79353622290037, "f1_macro": 92.54569249881214, "gmean": 91.36818236428712, "threshold": 0.4979080259799957}
